# DEEP R on multi-MNIST vs static block-sparse / dense baselines

A 1-hidden-layer MLP on 2-task multi-MNIST. Compares three training regimes:

1. **DEEP R**: dense init, L1 decay + Gaussian noise on the active weights, with
   sign-flip-based pruning. Rewiring (re-activating dormant weights) is optional
   and can be capped at a freeze step. When rewiring is enabled, the per-layer
   active-connection budget defaults to the block-sparse count.
2. **Static block-sparse**: each task uses its own half of the input pixels and
   half of the hidden units. No cross-task connections.
3. **Static dense**: standard fully-connected MLP.

Each baseline gets its own learning-rate sweep so the comparison is fair.

Metrics:
- **Train loss** (per-task softmax CE) over time, best-LR for each regime.
- **Active connections** in W1 and W2 over time (DEEP R).
- **Output purity**: for each output unit, the fraction of 2-hop paths
  `(input → hidden → output)` whose input belongs to the same task as the
  output. 1.0 = all paths within-task; 0.5 = uniform mix; 0 = all cross-task.


## Setup

In [1]:
import os
import sys

REPO_ROOT = '/home/edan/local_projects/phd_research'
for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'phd', 'structure_search')):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import jax
import jax.numpy as jnp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Multi-MNIST 2-task layout
N_TASKS = 2
NUM_CLASSES = 10
INPUT_PER_TASK = 784
INPUT_DIM = INPUT_PER_TASK * N_TASKS              # 1568
OUTPUT_DIM = NUM_CLASSES * N_TASKS                # 20

# 1-hidden-layer architecture
N_HIDDEN = 128
HIDDEN_PER_TASK = N_HIDDEN // N_TASKS             # 64

# Block-sparse param counts (per-layer connection budget for DEEP R rewiring).
BLOCK_SPARSE_W1 = INPUT_PER_TASK * HIDDEN_PER_TASK * N_TASKS    # 784*64*2 = 100352
BLOCK_SPARSE_W2 = HIDDEN_PER_TASK * NUM_CLASSES * N_TASKS       # 64*10*2  = 1280

print('JAX device:', jax.devices()[0])
print(f'INPUT_DIM={INPUT_DIM}  N_HIDDEN={N_HIDDEN}  OUTPUT_DIM={OUTPUT_DIM}')
print(f'Dense W1 params: {INPUT_DIM*N_HIDDEN}, block-sparse W1: {BLOCK_SPARSE_W1}')
print(f'Dense W2 params: {N_HIDDEN*OUTPUT_DIM}, block-sparse W2: {BLOCK_SPARSE_W2}')

JAX device: cuda:0
INPUT_DIM=1568  N_HIDDEN=128  OUTPUT_DIM=20
Dense W1 params: 200704, block-sparse W1: 100352
Dense W2 params: 2560, block-sparse W2: 1280


## Data

In [2]:
def load_data():
    """Load MNIST and standardize per-pixel (mean 0, std 1)."""
    from data import load_dataset
    images, labels, _, _ = load_dataset('mnist', split='train')
    images = np.asarray(images, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.int32)
    mean = images.mean(axis=0, keepdims=True)
    std = images.std(axis=0, keepdims=True)
    normalized = (images - mean) / np.maximum(std, 1e-3)
    return jnp.asarray(normalized), jnp.asarray(labels)


images, labels = load_data()
print('images:', images.shape, 'labels:', labels.shape)

images: (60000, 784) labels: (60000,)


## Architecture

Standard MLP: `x → W1 + b1 → ReLU → W2 + b2 → per-task softmax CE`.

In [3]:
def forward(W1, b1, W2, b2, x):
    z1 = x @ W1 + b1
    h = jax.nn.relu(z1)
    logits = h @ W2 + b2
    return logits, h, z1


def loss_fn(W1, b1, W2, b2, x, y):
    logits, _, _ = forward(W1, b1, W2, b2, x)
    logits_pt = logits.reshape(N_TASKS, NUM_CLASSES)
    lp = jax.nn.log_softmax(logits_pt, axis=-1)
    return -jnp.mean(jnp.sum(jax.nn.one_hot(y, NUM_CLASSES) * lp, axis=-1))


def make_sample(images, labels, key):
    k1, k2 = jax.random.split(key)
    idx1 = jax.random.randint(k1, (), 0, images.shape[0])
    idx2 = jax.random.randint(k2, (), 0, images.shape[0])
    x = jnp.concatenate([images[idx1], images[idx2]])
    y = jnp.array([labels[idx1], labels[idx2]])
    return x, y

## Init helpers and masks

In [4]:
def kaiming_init(key, shape, fan_in):
    bound = jnp.sqrt(3.0 / float(fan_in))
    return jax.random.uniform(key, shape, minval=-bound, maxval=bound)


def init_dense_params(seed=0):
    """Initialize a dense MLP. Returns (W1, b1, W2, b2)."""
    k = jax.random.key(seed)
    k1, k2 = jax.random.split(k)
    W1 = kaiming_init(k1, (INPUT_DIM, N_HIDDEN), INPUT_DIM)
    W2 = kaiming_init(k2, (N_HIDDEN, OUTPUT_DIM), N_HIDDEN)
    b1 = jnp.zeros((N_HIDDEN,))
    b2 = jnp.zeros((OUTPUT_DIM,))
    return W1, b1, W2, b2


def block_sparse_masks():
    """Disjoint per-task subnets. M1: input task == hidden task. M2: hidden task == output task."""
    input_task  = jnp.arange(INPUT_DIM) // INPUT_PER_TASK
    hidden_task = jnp.arange(N_HIDDEN) // HIDDEN_PER_TASK
    output_task = jnp.arange(OUTPUT_DIM) // NUM_CLASSES
    M1 = (input_task[:, None]  == hidden_task[None, :]).astype(jnp.float32)
    M2 = (hidden_task[:, None] == output_task[None, :]).astype(jnp.float32)
    return M1, M2


def dense_masks():
    return jnp.ones((INPUT_DIM, N_HIDDEN)), jnp.ones((N_HIDDEN, OUTPUT_DIM))


# Quick check that the block-sparse count matches our constants.
_M1, _M2 = block_sparse_masks()
print(f'block-sparse active: W1={int(_M1.sum())} (expected {BLOCK_SPARSE_W1}), '
      f'W2={int(_M2.sum())} (expected {BLOCK_SPARSE_W2})')

block-sparse active: W1=100352 (expected 100352), W2=1280 (expected 1280)


## Static training

Plain SGD with frozen masks. Used for the block-sparse and dense baselines.
JIT-compiles once; running again with a new `lr` reuses the cache.

In [5]:
def train_static(W1_init, b1_init, W2_init, b2_init,
                  M1, M2, images, labels,
                  lr, n_steps, snapshot_every, seed):
    n_chunks = n_steps // snapshot_every
    init_carry = (W1_init, b1_init, W2_init, b2_init,
                  jnp.array(0, dtype=jnp.int32))

    def step_fn(carry, key):
        W1, b1, W2, b2, t = carry
        x, y = make_sample(images, labels, key)
        loss, grads = jax.value_and_grad(loss_fn, argnums=(0, 1, 2, 3))(W1, b1, W2, b2, x, y)
        g1, gb1, g2, gb2 = grads
        W1 = W1 - lr * g1 * M1
        b1 = b1 - lr * gb1
        W2 = W2 - lr * g2 * M2
        b2 = b2 - lr * gb2
        return (W1, b1, W2, b2, t + 1), loss

    def chunk_fn(carry, key):
        keys = jax.random.split(key, snapshot_every)
        carry, losses = jax.lax.scan(step_fn, carry, keys)
        snap = dict(step=carry[-1], avg_loss=losses.mean())
        return carry, snap

    keys = jax.random.split(jax.random.key(seed), n_chunks)
    final_carry, snaps = jax.lax.scan(chunk_fn, init_carry, keys)
    out = {k: jax.device_get(v) for k, v in snaps.items()}
    out['final_W1'] = jax.device_get(final_carry[0])
    out['final_b1'] = jax.device_get(final_carry[1])
    out['final_W2'] = jax.device_get(final_carry[2])
    out['final_b2'] = jax.device_get(final_carry[3])
    return out


train_static_jit = jax.jit(
    train_static,
    static_argnames=('n_steps', 'snapshot_every', 'seed'),
)

## DEEP R training

Each weight is `w = s * theta`, with `s ∈ {-1, +1}` fixed when active and
`theta = |w| > 0`. We store the signed weight `W` and a boolean mask `M`
together — equivalent and simpler.

Per-step update on each active weight:
- `theta -= lr * (s * grad + l1) + sqrt(2*lr*T) * Z`,  Z ~ N(0, 1)
- equivalently  `W -= lr * (grad + l1 * sign(W)) - sqrt(2*lr*T) * Z` (after
  absorbing sign into the noise; symmetric).
- If `sign(new_W) != sign(W)` → deactivate (`M=0`, `W=0`). The connection
  becomes dormant.

Rewiring (optional, every chunk, only if `step < rewire_until`):
- For each layer with a non-None `target_active_*`, count active. If less than
  the target, reactivate `(target − active)` random dormant entries with a small
  random sign and `theta = init_scale`.

Biases never get sparsified.

In [6]:
def deep_r_step(carry, key, lr, l1, temperature, images, labels):
    W1, b1, W2, b2, M1, M2, t = carry
    data_key, n1_key, n2_key = jax.random.split(key, 3)
    x, y = make_sample(images, labels, data_key)
    loss, grads = jax.value_and_grad(loss_fn, argnums=(0, 1, 2, 3))(W1, b1, W2, b2, x, y)
    g1, gb1, g2, gb2 = grads

    noise_scale = jnp.sqrt(2.0 * lr * temperature)
    s1 = jnp.sign(W1)
    s2 = jnp.sign(W2)
    n1 = jax.random.normal(n1_key, W1.shape) * noise_scale
    n2 = jax.random.normal(n2_key, W2.shape) * noise_scale

    # Update only on currently-active weights.
    new_W1 = W1 - lr * g1 - lr * l1 * s1 + n1
    new_W2 = W2 - lr * g2 - lr * l1 * s2 + n2
    new_W1 = jnp.where(M1, new_W1, 0.0)
    new_W2 = jnp.where(M2, new_W2, 0.0)

    # Sign-flip → deactivate. M stays True iff sign matches AND was active.
    new_M1 = M1 & (jnp.sign(new_W1) == s1) & (s1 != 0)
    new_M2 = M2 & (jnp.sign(new_W2) == s2) & (s2 != 0)
    new_W1 = jnp.where(new_M1, new_W1, 0.0)
    new_W2 = jnp.where(new_M2, new_W2, 0.0)

    b1 = b1 - lr * gb1
    b2 = b2 - lr * gb2
    return (new_W1, b1, new_W2, b2, new_M1, new_M2, t + 1), loss


def deep_r_chunk_factory(lr, l1, temperature, images, labels, chunk_steps):
    def chunk_fn(carry, keys):
        def _step(carry, key):
            return deep_r_step(carry, key, lr, l1, temperature, images, labels)
        carry, losses = jax.lax.scan(_step, carry, keys)
        return carry, losses.mean()
    return jax.jit(chunk_fn)


def rewire_layer(M, W, target, key, init_scale=1e-3):
    """Re-activate (target - n_active) random dormant entries with a random sign and small magnitude."""
    M_np = np.asarray(M).astype(bool)
    W_np = np.asarray(W).copy()
    n_active = int(M_np.sum())
    n_to_add = target - n_active
    if n_to_add <= 0:
        return jnp.asarray(M_np.astype(jnp.float32) if M.dtype != jnp.bool_ else M_np), W

    flat_M = M_np.reshape(-1)
    dormant_idx = np.flatnonzero(~flat_M)
    if dormant_idx.size == 0:
        return M, W

    n_to_add = min(n_to_add, dormant_idx.size)
    k_perm, k_sign = jax.random.split(key)
    perm = np.asarray(jax.random.permutation(k_perm, dormant_idx.size))
    chosen = dormant_idx[perm[:n_to_add]]
    signs = np.asarray(
        jax.random.choice(k_sign, jnp.array([-1.0, 1.0]), shape=(n_to_add,))
    )

    flat_W = W_np.reshape(-1)
    flat_W[chosen] = signs * init_scale
    flat_M[chosen] = True
    return jnp.asarray(flat_M.reshape(M.shape)), jnp.asarray(flat_W.reshape(W.shape))


def train_deep_r(W1_init, b1_init, W2_init, b2_init,
                 M1_init, M2_init, images, labels, *,
                 lr=2**-7,
                 l1=1e-5,
                 temperature=1e-7,
                 rewire=False,
                 target_W1=None,
                 target_W2=None,
                 rewire_until=None,
                 init_scale=1e-3,
                 n_steps=50_000,
                 chunk_steps=1_000,
                 seed=0):
    """DEEP R training. Returns dict with per-chunk snapshots.

    rewire: if True, after each chunk reactivate up to `target_W{1,2}` connections.
    rewire_until: int step beyond which no further rewiring (None = never stop).
    """
    chunk_fn = deep_r_chunk_factory(lr, l1, temperature, images, labels, chunk_steps)

    n_chunks = n_steps // chunk_steps
    rng = jax.random.key(seed)

    W1, b1, W2, b2 = W1_init, b1_init, W2_init, b2_init
    M1 = M1_init.astype(jnp.bool_)
    M2 = M2_init.astype(jnp.bool_)

    snaps = {'step': [], 'avg_loss': [], 'n_active_W1': [], 'n_active_W2': [],
             'M1': [], 'M2': []}

    for chunk_i in range(n_chunks):
        rng, k_chunk, k_rew1, k_rew2 = jax.random.split(rng, 4)
        keys = jax.random.split(k_chunk, chunk_steps)
        carry = (W1, b1, W2, b2, M1, M2, jnp.array(chunk_i * chunk_steps, dtype=jnp.int32))
        carry, avg_loss = chunk_fn(carry, keys)
        W1, b1, W2, b2, M1, M2, _ = carry
        step = (chunk_i + 1) * chunk_steps

        if rewire and (rewire_until is None or step <= rewire_until):
            if target_W1 is not None:
                M1, W1 = rewire_layer(M1, W1, target_W1, k_rew1, init_scale=init_scale)
            if target_W2 is not None:
                M2, W2 = rewire_layer(M2, W2, target_W2, k_rew2, init_scale=init_scale)

        snaps['step'].append(step)
        snaps['avg_loss'].append(float(avg_loss))
        snaps['n_active_W1'].append(int(jnp.sum(M1)))
        snaps['n_active_W2'].append(int(jnp.sum(M2)))
        snaps['M1'].append(np.asarray(M1, dtype=bool))
        snaps['M2'].append(np.asarray(M2, dtype=bool))

    snaps = {k: (np.asarray(v) if k != 'M1' and k != 'M2' else np.stack(v))
             for k, v in snaps.items()}
    snaps['final_W1'] = np.asarray(W1)
    snaps['final_b1'] = np.asarray(b1)
    snaps['final_W2'] = np.asarray(W2)
    snaps['final_b2'] = np.asarray(b2)
    return snaps

## Metrics

**Output purity**: for each output `o`, count 2-hop paths `(i, h, o)` where
both `M1[i, h]` and `M2[h, o]` are active. The path is *within-task* if input
`i` lies in the same task as output `o`. Purity = within / total.

In [7]:
INPUT_TASK  = np.arange(INPUT_DIM)  // INPUT_PER_TASK
OUTPUT_TASK = np.arange(OUTPUT_DIM) // NUM_CLASSES
SAME_TASK_IO = (INPUT_TASK[:, None] == OUTPUT_TASK[None, :])   # (IN, OUT) bool


def output_purity(M1, M2):
    """Returns (per-output purity, same-task-paths, total-paths). All shape (OUT,)."""
    M1f = np.asarray(M1, dtype=np.float32)
    M2f = np.asarray(M2, dtype=np.float32)
    P = M1f @ M2f                                       # (IN, OUT) path counts
    same  = (P * SAME_TASK_IO).sum(axis=0)              # (OUT,)
    total = P.sum(axis=0)                               # (OUT,)
    purity = np.where(total > 0, same / np.maximum(total, 1), np.nan)
    return purity, same, total


def purity_over_time(snaps):
    """Returns (steps, mean_purity, per_output_purity). per_output: (n_snap, OUT)."""
    n = len(snaps['step'])
    per_out = np.zeros((n, OUTPUT_DIM))
    for i in range(n):
        p, _, _ = output_purity(snaps['M1'][i], snaps['M2'][i])
        per_out[i] = p
    mean_p = np.nanmean(per_out, axis=-1)
    return np.asarray(snaps['step']), mean_p, per_out

## LR sweep — block-sparse baseline

Quick sweep of learning rates on the static block-sparse network. Best LR is
the one with the lowest final-window mean loss.

In [8]:
# A coarse log-spaced LR grid. Adjust as needed.
LR_GRID = [2**-12, 2**-10, 2**-8, 2**-7]
N_STEPS = 2_000_000
SNAPSHOT_EVERY = 1_000
TAIL_WINDOW = 5     # average over last 5 snapshots = last 5k steps


def best_lr_run(results):
    """Pick the LR with the lowest tail-window mean loss; return (lr, run)."""
    scored = [(np.mean(r['avg_loss'][-TAIL_WINDOW:]), lr, r) for lr, r in results.items()
              if np.isfinite(r['avg_loss'][-1])]
    scored.sort(key=lambda x: x[0])
    return scored[0][1], scored[0][2], scored


def run_static_lr_sweep(M1, M2, lrs=LR_GRID, n_steps=N_STEPS, seed=0, label=''):
    W1, b1, W2, b2 = init_dense_params(seed=seed)
    # Apply mask to init weights so we don't start with disallowed connections.
    W1 = W1 * M1
    W2 = W2 * M2
    out = {}
    for lr in lrs:
        snaps = train_static_jit(W1, b1, W2, b2, M1, M2, images, labels,
                                  lr=jnp.float32(lr),
                                  n_steps=n_steps,
                                  snapshot_every=SNAPSHOT_EVERY,
                                  seed=seed)
        tail = float(np.mean(snaps['avg_loss'][-TAIL_WINDOW:]))
        print(f'  [{label}] lr=2**{np.log2(lr):.1f}: tail_loss={tail:.4f}')
        out[lr] = snaps
    return out


M1_block, M2_block = block_sparse_masks()
block_sweep = run_static_lr_sweep(M1_block, M2_block, label='block-sparse')
best_lr_block, best_block, _ = best_lr_run(block_sweep)
print(f'\nBest block-sparse LR: 2**{np.log2(best_lr_block):.1f}, '
      f'tail loss {float(np.mean(best_block["avg_loss"][-TAIL_WINDOW:])):.4f}')

KeyboardInterrupt: 

## LR sweep — dense FC baseline

In [ ]:
M1_dense, M2_dense = dense_masks()
fc_sweep = run_static_lr_sweep(M1_dense, M2_dense, label='dense-FC')
best_lr_fc, best_fc, _ = best_lr_run(fc_sweep)
print(f'\nBest dense-FC LR: 2**{np.log2(best_lr_fc):.1f}, '
      f'tail loss {float(np.mean(best_fc["avg_loss"][-TAIL_WINDOW:])):.4f}')

  [dense-FC] lr=2**-12.0: tail_loss=0.0883
  [dense-FC] lr=2**-10.0: tail_loss=0.0379
  [dense-FC] lr=2**-8.0: tail_loss=0.0728
  [dense-FC] lr=2**-7.0: tail_loss=2814709770223616.0000

Best dense-FC LR: 2**-10.0, tail loss 0.0379


## Run DEEP R

Edit the cell to flip rewiring on/off, change the budget, or set a freeze
step. With `rewire=False`, the network starts dense and prunes to whatever
equilibrium the L1 / gradient balance finds.

`temperature` controls the Gaussian-noise std. Set to 0 for deterministic
pruning. Bellec et al. used ~`1e-5`.

In [ ]:
N_STEPS = 15_000_000

# DEEP R config — modify freely.
DEEP_R_CONFIG = dict(
    lr=2**-10, # best_lr_fc,                 # default to FC's best LR; sweep separately if you want
    l1=1e-5,
    temperature=1e-7,              # gradient-noise std
    rewire=False,                  # if True, maintain target budgets via rewiring
    target_W1=BLOCK_SPARSE_W1,     # used only if rewire=True
    target_W2=BLOCK_SPARSE_W2,
    rewire_until=None,             # int step to stop rewiring (None = never stop)
    init_scale=1e-3,               # |theta| of newly-rewired entries
    n_steps=N_STEPS,
    chunk_steps=SNAPSHOT_EVERY,
    seed=0,
)


def run_deep_r(config):
    W1, b1, W2, b2 = init_dense_params(seed=config['seed'])
    M1, M2 = dense_masks()
    return train_deep_r(W1, b1, W2, b2, M1, M2, images, labels, **config)


deep_r_snaps = run_deep_r(DEEP_R_CONFIG)
print(f'DEEP R: tail loss = {float(np.mean(deep_r_snaps["avg_loss"][-TAIL_WINDOW:])):.4f}')
print(f'        final active: W1={deep_r_snaps["n_active_W1"][-1]} '
      f'(/{INPUT_DIM*N_HIDDEN}), W2={deep_r_snaps["n_active_W2"][-1]} (/{N_HIDDEN*OUTPUT_DIM})')

## Plots

In [ ]:
def plot_loss_comparison():
    """Best-LR loss curves for all three regimes."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=np.arange(len(best_block['avg_loss'])) * SNAPSHOT_EVERY,
        y=best_block['avg_loss'],
        mode='lines', name=f'block-sparse (lr=2^{np.log2(best_lr_block):.1f})'))
    fig.add_trace(go.Scatter(
        x=np.arange(len(best_fc['avg_loss'])) * SNAPSHOT_EVERY,
        y=best_fc['avg_loss'],
        mode='lines', name=f'dense FC (lr=2^{np.log2(best_lr_fc):.1f})'))
    fig.add_trace(go.Scatter(
        x=deep_r_snaps['step'],
        y=deep_r_snaps['avg_loss'],
        mode='lines', name=f'DEEP R (lr=2^{np.log2(DEEP_R_CONFIG["lr"]):.1f}, rewire={DEEP_R_CONFIG["rewire"]})'))
    fig.update_layout(title='Train loss — best LR per regime',
                      xaxis_title='step', yaxis_title='per-task softmax CE',
                      width=900, height=420)
    fig.show()
    return fig


plot_loss_comparison()

NameError: name 'best_block' is not defined

In [ ]:
def plot_lr_sweep():
    """Tail loss vs LR for both static sweeps."""
    lrs = sorted(block_sweep.keys())
    block_tail = [float(np.mean(block_sweep[lr]['avg_loss'][-TAIL_WINDOW:])) for lr in lrs]
    fc_tail    = [float(np.mean(fc_sweep[lr]['avg_loss'][-TAIL_WINDOW:])) for lr in lrs]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=[np.log2(lr) for lr in lrs], y=block_tail,
                             mode='lines+markers', name='block-sparse'))
    fig.add_trace(go.Scatter(x=[np.log2(lr) for lr in lrs], y=fc_tail,
                             mode='lines+markers', name='dense FC'))
    fig.update_layout(title=f'Tail loss (avg of last {TAIL_WINDOW} snapshots) vs LR',
                      xaxis_title='log2(lr)', yaxis_title='tail loss',
                      width=800, height=380)
    fig.show()
    return fig


plot_lr_sweep()

In [ ]:
def plot_deep_r_active():
    """Active connection counts in W1, W2 over training."""
    steps = deep_r_snaps['step']
    fig = make_subplots(rows=1, cols=2, subplot_titles=['W1 (input→hidden)', 'W2 (hidden→output)'])
    fig.add_trace(go.Scatter(x=steps, y=deep_r_snaps['n_active_W1'], mode='lines',
                             name='W1 active'), row=1, col=1)
    fig.add_hline(y=BLOCK_SPARSE_W1, line_dash='dash',
                  annotation_text=f'block-sparse budget ({BLOCK_SPARSE_W1})',
                  row=1, col=1)
    fig.add_hline(y=INPUT_DIM*N_HIDDEN, line_dash='dot',
                  annotation_text=f'dense ({INPUT_DIM*N_HIDDEN})',
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=steps, y=deep_r_snaps['n_active_W2'], mode='lines',
                             name='W2 active'), row=1, col=2)
    fig.add_hline(y=BLOCK_SPARSE_W2, line_dash='dash',
                  annotation_text=f'block-sparse budget ({BLOCK_SPARSE_W2})',
                  row=1, col=2)
    fig.add_hline(y=N_HIDDEN*OUTPUT_DIM, line_dash='dot',
                  annotation_text=f'dense ({N_HIDDEN*OUTPUT_DIM})',
                  row=1, col=2)
    fig.update_xaxes(title_text='step')
    fig.update_yaxes(title_text='# active connections')
    fig.update_layout(title='DEEP R: active connections over time',
                      height=420, width=900, showlegend=False)
    fig.show()
    return fig


plot_deep_r_active()

NameError: name 'deep_r_snaps' is not defined

In [ ]:
def plot_purity_over_time():
    steps, mean_p, per_out = purity_over_time(deep_r_snaps)
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=['mean output purity', 'per-output purity'])
    fig.add_trace(go.Scatter(x=steps, y=mean_p, mode='lines', name='mean'),
                  row=1, col=1)
    fig.add_hline(y=0.5, line_dash='dot', annotation_text='chance (0.5)',
                  row=1, col=1)
    for o in range(OUTPUT_DIM):
        task = 'task 0' if o < NUM_CLASSES else 'task 1'
        color = '#1f77b4' if o < NUM_CLASSES else '#d62728'
        fig.add_trace(go.Scatter(x=steps, y=per_out[:, o], mode='lines',
                                 name=f'out {o} ({task})',
                                 line=dict(color=color, width=1),
                                 opacity=0.5, showlegend=False),
                      row=1, col=2)
    fig.update_xaxes(title_text='step')
    fig.update_yaxes(title_text='purity', range=[0, 1.05])
    fig.update_layout(title='DEEP R: output purity over time',
                      height=420, width=900)
    fig.show()
    return fig


plot_purity_over_time()

In [ ]:
def plot_final_purity_bars():
    purity, same, total = output_purity(deep_r_snaps['M1'][-1], deep_r_snaps['M2'][-1])
    colors = ['#1f77b4' if o < NUM_CLASSES else '#d62728' for o in range(OUTPUT_DIM)]
    fig = go.Figure()
    fig.add_trace(go.Bar(x=list(range(OUTPUT_DIM)), y=purity,
                         marker_color=colors,
                         hovertext=[f'output {o}: {int(same[o])}/{int(total[o])} same-task paths'
                                    for o in range(OUTPUT_DIM)]))
    fig.add_hline(y=0.5, line_dash='dot')
    fig.update_layout(title='DEEP R: final per-output purity (blue=task 0, red=task 1)',
                      xaxis_title='output unit', yaxis_title='purity',
                      yaxis_range=[0, 1.05],
                      width=900, height=380)
    fig.show()
    return fig


plot_final_purity_bars()